# 智侠 (ZhiXia) - 集成测试Notebook

这个notebook整合了所有测试和功能演示，包括：
- **基础配置**：环境变量、库加载、模型路径
- **LLM推理测试**：RKLLM NPU推理
- **完整管道**：ASR → LLM → TTS工作流
- **工具函数**：垃圾回收、音频处理

按照能力级别从上到下运行，每个段落独立且有说明。

## 第1部分：基础配置与环境初始化

In [ ]:
"""基础配置 - 环境变量和库初始化"""
import os
import sys
import gc
import json
import time
import ctypes
from typing import Optional, List, Dict
from dataclasses import dataclass
from enum import IntEnum
from pathlib import Path

# 获取脚本目录（适配Notebook和本地路径）
if 'get_ipython' in dir():
    # Jupyter环境
    script_dir = os.getcwd()
else:
    # 本地Python脚本
    script_dir = os.path.dirname(os.path.abspath(__file__))

print(f"📁 工作目录: {script_dir}")

# 自动配置环境变量
os.environ['MODELSCOPE_CACHE'] = os.path.join(script_dir, '.cache', 'modelscope')
os.environ['HOME'] = script_dir
os.environ['PYTHONPATH'] = os.path.join(script_dir, '.local', 'lib', 'python3.9', 'site-packages') + ':' + os.environ.get('PYTHONPATH', '')
os.environ['LD_LIBRARY_PATH'] = os.path.join(script_dir, 'rknn_libs') + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# 确保必要目录存在
essential_dirs = [
    os.path.join(script_dir, '.cache', 'modelscope'),
    os.path.join(script_dir, 'models'),
    os.path.join(script_dir, 'output'),
    os.path.join(script_dir, 'rknn_libs')
]

for dir_path in essential_dirs:
    os.makedirs(dir_path, exist_ok=True)
    exists = "✅" if os.path.exists(dir_path) else "❌"
    print(f"  {exists} {dir_path}")

## 第2部分：RKLLM NPU推理引擎

In [ ]:
"""RKLLM核心模块 - ctypes包装器和配置"""

# 模型类型常量
MODEL_TYPE_QWEN2 = "qwen2"
MODEL_TYPE_QWEN3 = "qwen3"

# 加载RKLLM Runtime库
rkllm_lib_path = os.path.join(script_dir, 'rknn_libs', 'librkllmrt.so')

if not os.path.exists(rkllm_lib_path):
    # 尝试其他路径
    rkllm_lib_path = '/usr/lib/librkllmrt.so'
    if not os.path.exists(rkllm_lib_path):
        rkllm_lib_path = os.path.join(script_dir, 'librkllmrt.so')

_rkllm_lib = None
try:
    _rkllm_lib = ctypes.CDLL(rkllm_lib_path)
    print(f"✅ 成功加载RKLLM库: {rkllm_lib_path}")
except OSError as e:
    print(f"⚠️  无法加载RKLLM库: {e}")
    print(f"   查找路径: {rkllm_lib_path}")
    print(f"   (在RK3588板子上会自动加载，开发机可忽略)")

In [ ]:
"""RKLLM类型定义 - C结构体和枚举"""

class LLMCallState(IntEnum):
    """LLM调用状态"""
    RUN_NORMAL = 0
    RUN_WAITING = 1
    RUN_FINISH = 2
    RUN_ERROR = 3


class RKLLMInputType(IntEnum):
    """输入类型"""
    PROMPT = 0
    TOKEN = 1
    EMBED = 2
    MULTIMODAL = 3


class RKLLMInferMode(IntEnum):
    """推理模式"""
    GENERATE = 0
    GET_LAST_HIDDEN_LAYER = 1
    GET_LOGITS = 2


# 定义C结构体
class RKLLMExtendParam(ctypes.Structure):
    _fields_ = [
        ("base_domain_id", ctypes.c_int32),
        ("embed_flash", ctypes.c_int8),
        ("enabled_cpus_num", ctypes.c_int8),
        ("enabled_cpus_mask", ctypes.c_uint32),
        ("n_batch", ctypes.c_uint8),
        ("use_cross_attn", ctypes.c_int8),
        ("reserved", ctypes.c_uint8 * 104),
    ]


class RKLLMParam(ctypes.Structure):
    _fields_ = [
        ("model_path", ctypes.c_char_p),
        ("max_context_len", ctypes.c_int32),
        ("max_new_tokens", ctypes.c_int32),
        ("top_k", ctypes.c_int32),
        ("n_keep", ctypes.c_int32),
        ("top_p", ctypes.c_float),
        ("temperature", ctypes.c_float),
        ("repeat_penalty", ctypes.c_float),
        ("frequency_penalty", ctypes.c_float),
        ("presence_penalty", ctypes.c_float),
        ("mirostat", ctypes.c_int32),
        ("mirostat_tau", ctypes.c_float),
        ("mirostat_eta", ctypes.c_float),
        ("skip_special_token", ctypes.c_bool),
        ("is_async", ctypes.c_bool),
        ("img_start", ctypes.c_char_p),
        ("img_end", ctypes.c_char_p),
        ("img_content", ctypes.c_char_p),
        ("extend_param", RKLLMExtendParam),
    ]


class RKLLMInputUnion(ctypes.Union):
    _fields_ = [
        ("prompt_input", ctypes.c_char_p),
        ("embed_input", ctypes.c_void_p),
        ("token_input", ctypes.c_void_p),
        ("multimodal_input", ctypes.c_void_p),
    ]


class RKLLMInput(ctypes.Structure):
    _fields_ = [
        ("role", ctypes.c_char_p),
        ("enable_thinking", ctypes.c_bool),
        ("input_type", ctypes.c_int),
        ("input", RKLLMInputUnion),
    ]


class RKLLMInferParam(ctypes.Structure):
    _fields_ = [
        ("mode", ctypes.c_int),
        ("lora_params", ctypes.c_void_p),
        ("prompt_cache_params", ctypes.c_void_p),
        ("keep_history", ctypes.c_int),
    ]


class RKLLMPerfStat(ctypes.Structure):
    _fields_ = [
        ("prefill_time_ms", ctypes.c_float),
        ("prefill_tokens", ctypes.c_int),
        ("generate_time_ms", ctypes.c_float),
        ("generate_tokens", ctypes.c_int),
        ("memory_usage_mb", ctypes.c_float),
    ]


class RKLLMResult(ctypes.Structure):
    _fields_ = [
        ("text", ctypes.c_char_p),
        ("token_id", ctypes.c_int32),
        ("last_hidden_layer", ctypes.c_void_p),
        ("logits", ctypes.c_void_p),
        ("perf", RKLLMPerfStat),
    ]


# 回调函数类型
LLMResultCallback = ctypes.CFUNCTYPE(
    ctypes.c_int,
    ctypes.POINTER(RKLLMResult),
    ctypes.c_void_p,
    ctypes.c_int
)

print("✅ RKLLM类型结构定义完成")

In [ ]:
"""RKLLM配置类"""

@dataclass
class RKLLMConfig:
    """RKLLM推理配置"""
    model_path: str
    max_context_len: int = 4096
    max_new_tokens: int = 256
    top_k: int = 40
    top_p: float = 0.9
    temperature: float = 0.7
    repeat_penalty: float = 1.1
    frequency_penalty: float = 0.0
    presence_penalty: float = 0.0
    skip_special_token: bool = True
    model_type: str = "auto"  # auto, qwen2, qwen3
    enable_thinking: bool = False  # Qwen3思考模式
    
    def __repr__(self):
        return f"""RKLLMConfig(
  模型路径: {self.model_path}
  模型类型: {self.model_type}
  最大上下文: {self.max_context_len}
  最大生成token数: {self.max_new_tokens}
  温度: {self.temperature}
  top_p: {self.top_p}
)"""

print("✅ RKLLMConfig类定义完成")

In [ ]:
"""RKLLM推理类 - 核心功能"""

class RKLLM:
    """RKLLM NPU推理引擎"""
    
    def __init__(self, config: RKLLMConfig):
        if _rkllm_lib is None:
            raise RuntimeError("RKLLM库未加载，无法初始化 (仅在RK3588上可用)")
        
        self.config = config
        self.handle = ctypes.c_void_p()
        self._callback = None
        self._result_buffer = []
        
        # 自动检测模型类型
        if self.config.model_type == "auto":
            self.config.model_type = self._detect_model_type()
        
        # 创建默认参数
        self._create_default_param = _rkllm_lib.rkllm_createDefaultParam
        self._create_default_param.restype = RKLLMParam
        
        # 初始化函数
        self._init = _rkllm_lib.rkllm_init
        self._init.argtypes = [ctypes.POINTER(ctypes.c_void_p), ctypes.POINTER(RKLLMParam), LLMResultCallback]
        self._init.restype = ctypes.c_int
        
        # 运行函数
        self._run = _rkllm_lib.rkllm_run
        self._run.argtypes = [ctypes.c_void_p, ctypes.POINTER(RKLLMInput), ctypes.POINTER(RKLLMInferParam), ctypes.c_void_p]
        self._run.restype = ctypes.c_int
        
        # 销毁函数
        self._destroy = _rkllm_lib.rkllm_destroy
        self._destroy.argtypes = [ctypes.c_void_p]
        self._destroy.restype = ctypes.c_int
        
        # 设置chat template
        self._set_chat_template = _rkllm_lib.rkllm_set_chat_template
        self._set_chat_template.argtypes = [ctypes.c_void_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
        self._set_chat_template.restype = ctypes.c_int
        
        self._init_model()
    
    def _detect_model_type(self) -> str:
        """从模型文件名检测模型类型"""
        model_name = os.path.basename(self.config.model_path).lower()
        if "qwen3" in model_name:
            return MODEL_TYPE_QWEN3
        elif "qwen2" in model_name or "qwen-2" in model_name:
            return MODEL_TYPE_QWEN2
        else:
            return MODEL_TYPE_QWEN2
    
    def _result_callback(self, result_ptr, userdata, state):
        """结果回调函数"""
        if result_ptr:
            result = result_ptr.contents
            if result.text:
                text = result.text.decode('utf-8')
                self._result_buffer.append(text)
                
                if state == LLMCallState.RUN_FINISH:
                    perf = result.perf
                    print(f"\n[性能统计] Prefill: {perf.prefill_time_ms:.2f}ms ({perf.prefill_tokens} tokens), "
                          f"Generate: {perf.generate_time_ms:.2f}ms ({perf.generate_tokens} tokens), "
                          f"Memory: {perf.memory_usage_mb:.2f}MB")
        
        return 0
    
    def _init_model(self):
        """初始化模型"""
        param = self._create_default_param()
        
        param.model_path = self.config.model_path.encode('utf-8')
        param.max_context_len = self.config.max_context_len
        param.max_new_tokens = self.config.max_new_tokens
        param.top_k = self.config.top_k
        param.top_p = self.config.top_p
        param.temperature = self.config.temperature
        param.repeat_penalty = self.config.repeat_penalty
        param.frequency_penalty = self.config.frequency_penalty
        param.presence_penalty = self.config.presence_penalty
        param.skip_special_token = self.config.skip_special_token
        param.is_async = False
        
        param.extend_param.enabled_cpus_num = 4
        param.extend_param.enabled_cpus_mask = 0x0F
        
        self._callback = LLMResultCallback(self._result_callback)
        
        ret = self._init(ctypes.byref(self.handle), ctypes.byref(param), self._callback)
        if ret != 0:
            raise RuntimeError(f"RKLLM初始化失败，错误码: {ret}")
        
        print(f"✅ RKLLM模型初始化成功: {self.config.model_path}")
    
    def generate(self, prompt: str, role: str = "user") -> str:
        """生成文本"""
        self._result_buffer = []
        
        input_data = RKLLMInput()
        input_data.role = role.encode('utf-8')
        input_data.enable_thinking = False
        input_data.input_type = RKLLMInputType.PROMPT
        input_data.input.prompt_input = prompt.encode('utf-8')
        
        infer_param = RKLLMInferParam()
        infer_param.mode = RKLLMInferMode.GENERATE
        infer_param.lora_params = None
        infer_param.prompt_cache_params = None
        infer_param.keep_history = 1
        
        ret = self._run(self.handle, ctypes.byref(input_data), ctypes.byref(infer_param), None)
        if ret != 0:
            raise RuntimeError(f"RKLLM推理失败，错误码: {ret}")
        
        return ''.join(self._result_buffer)
    
    def chat(self, messages: list, max_new_tokens: Optional[int] = None) -> str:
        """对话模式 - 根据模型类型选择不同的chat template"""
        if self.config.model_type == MODEL_TYPE_QWEN3:
            return self._chat_qwen3(messages, max_new_tokens)
        else:
            return self._chat_qwen2(messages, max_new_tokens)
    
    def _chat_qwen2(self, messages: list, max_new_tokens: Optional[int] = None) -> str:
        """Qwen2格式对话"""
        prompt_parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "system":
                prompt_parts.append(f"<|system|>\n{content}")
            elif role == "user":
                prompt_parts.append(f"<|user|>\n{content}")
            elif role == "assistant":
                prompt_parts.append(f"<|assistant|>\n{content}")
        
        prompt_parts.append("<|assistant|>\n")
        prompt = "\n".join(prompt_parts)
        
        return self.generate(prompt)
    
    def _chat_qwen3(self, messages: list, max_new_tokens: Optional[int] = None) -> str:
        """Qwen3格式对话 - 使用思考模式"""
        prompt_parts = ['<|im_start|>']
        
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            
            if role == "system":
                prompt_parts.append(f'<|im_start|>system\n{content}<|im_end|>')
            elif role == "user":
                prompt_parts.append(f'<|im_start|>user\n{content}<|im_end|>')
            elif role == "assistant":
                prompt_parts.append(f'<|im_start|>assistant\n{content}<|im_end|>')
            elif role == "tool":
                prompt_parts.append(f'<|im_start|>tool\n{content}<|im_end|>')
        
        thinking_content = "" if not self.config.enable_thinking else "<think>\n"
        prompt_parts.append(f'<|im_start|>assistant\n{thinking_content}')
        prompt = "".join(prompt_parts)
        
        return self._generate_with_thinking(prompt, self.config.enable_thinking)
    
    def _generate_with_thinking(self, prompt: str, enable_thinking: bool = False) -> str:
        """生成文本（支持思考模式）"""
        self._result_buffer = []
        
        input_data = RKLLMInput()
        input_data.role = b"user"
        input_data.enable_thinking = enable_thinking
        input_data.input_type = RKLLMInputType.PROMPT
        input_data.input.prompt_input = prompt.encode('utf-8')
        
        infer_param = RKLLMInferParam()
        infer_param.mode = RKLLMInferMode.GENERATE
        infer_param.lora_params = None
        infer_param.prompt_cache_params = None
        infer_param.keep_history = 1
        
        ret = self._run(self.handle, ctypes.byref(input_data), ctypes.byref(infer_param), None)
        if ret != 0:
            raise RuntimeError(f"RKLLM推理失败，错误码: {ret}")
        
        return ''.join(self._result_buffer)
    
    def __del__(self):
        """析构函数，释放资源"""
        if hasattr(self, 'handle') and self.handle:
            self._destroy(self.handle)
            print("✅ RKLLM资源已释放")


def create_rkllm_from_hf(model_path: str, **kwargs) -> RKLLM:
    """从模型路径创建RKLLM实例"""
    # 查找.rkllm文件
    if os.path.isdir(model_path):
        rkllm_files = [f for f in os.listdir(model_path) if f.endswith('.rkllm')]
        if rkllm_files:
            model_path = os.path.join(model_path, rkllm_files[0])
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"模型文件不存在: {model_path}")
    
    # 自动检测模型类型
    if 'model_type' not in kwargs:
        model_name = os.path.basename(model_path).lower()
        if 'qwen3' in model_name:
            kwargs['model_type'] = MODEL_TYPE_QWEN3
            print(f"检测到Qwen3模型: {model_path}")
        elif 'qwen2' in model_name or 'qwen-2' in model_name:
            kwargs['model_type'] = MODEL_TYPE_QWEN2
            print(f"检测到Qwen2模型: {model_path}")
    
    config = RKLLMConfig(model_path=model_path, **kwargs)
    return RKLLM(config)


print("✅ RKLLM推理类定义完成")

## 第3部分：工具函数

In [ ]:
"""辅助工具函数"""

def force_gc():
    """强制垃圾回收 - 释放内存给其他模块使用"""
    gc.collect()
    gc.collect()


def print_section(title: str, max_width: int = 60):
    """打印分隔符"""
    print("")
    print("=" * max_width)
    print(f"  {title}")
    print("=" * max_width)


def check_file_exists(file_path: str) -> bool:
    """检查文件是否存在"""
    exists = os.path.exists(file_path)
    status = "✅" if exists else "❌"
    print(f"{status} {file_path}")
    return exists


def list_models(models_dir: str = None) -> List[str]:
    """列出可用的RKLLM模型"""
    if models_dir is None:
        models_dir = os.path.join(script_dir, 'models')
    
    if not os.path.exists(models_dir):
        print(f"❌ 模型目录不存在: {models_dir}")
        return []
    
    models = [f for f in os.listdir(models_dir) if f.endswith('.rkllm')]
    if models:
        print(f"\n📁 找到 {len(models)} 个RKLLM模型:")
        for i, model in enumerate(models, 1):
            full_path = os.path.join(models_dir, model)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"  {i}. {model} ({size_mb:.2f} MB)")
    else:
        print(f"❌ 模型目录为空: {models_dir}")
    
    return [os.path.join(models_dir, m) for m in models]


print("✅ 工具函数定义完成")

## 第4部分：LLM推理测试

### 4.1 模型检测

In [ ]:
print_section("可用模型检测", 60)
available_models = list_models()

if not available_models:
    print("\n⚠️  未找到RKLLM模型文件")
    print("   请确保models目录中有.rkllm文件")
    print("   或手动指定模型路径")

### 4.2 简单推理测试（Qwen2）

In [ ]:
"""测试1: 简单文本生成 (Qwen2)"""

if available_models and _rkllm_lib is not None:
    print_section("测试1: Qwen2简单生成模式", 60)
    
    try:
        # 尝试找Qwen2模型
        model_path = None
        for model in available_models:
            if 'qwen2' in model.lower() and 'qwen3' not in model.lower():
                model_path = model
                break
        
        if model_path is None and available_models:
            model_path = available_models[0]
        
        if model_path:
            print(f"\n加载模型: {os.path.basename(model_path)}")
            llm = create_rkllm_from_hf(model_path, max_new_tokens=128)
            
            prompt = "你好，请介绍一下自己。"
            print(f"\n输入: {prompt}")
            print("\n生成中...")
            
            response = llm.generate(prompt)
            print(f"\n回复: {response}")
            
            del llm
            force_gc()
            time.sleep(0.5)
            
            print("\n✅ 测试1完成")
        else:
            print("❌ 没有可用的模型")
    except Exception as e:
        print(f"\n❌ 测试失败: {e}")
else:
    print("⚠️  跳过此测试（无模型或无RKLLM库）")

### 4.3 对话模式测试

In [ ]:
"""测试2: 对话模式"""

if available_models and _rkllm_lib is not None:
    print_section("测试2: 对话模式", 60)
    
    try:
        model_path = available_models[0] if available_models else None
        
        if model_path:
            print(f"\n加载模型: {os.path.basename(model_path)}")
            llm = create_rkllm_from_hf(model_path, max_new_tokens=128)
            
            messages = [
                {"role": "system", "content": "你是一个helpful的AI助手，请用简洁的方式回答问题。"},
                {"role": "user", "content": "什么是人工智能？"}
            ]
            
            print(f"\n系统提示: {messages[0]['content']}")
            print(f"用户提问: {messages[1]['content']}")
            print("\n生成中...")
            
            response = llm.chat(messages)
            print(f"\n助手回复: {response}")
            
            del llm
            force_gc()
            time.sleep(0.5)
            
            print("\n✅ 测试2完成")
        else:
            print("❌ 没有可用的模型")
    except Exception as e:
        print(f"\n❌ 测试失败: {e}")
else:
    print("⚠️  跳过此测试（无模型或无RKLLM库）")

## 第5部分：完整管道（ASR + LLM + TTS）

### 5.1 ASR (语音识别) 模块

In [ ]:
"""ASR模块 - FunASR语音识别"""

def asr_recognition_int8(audio_path: str) -> str:
    """使用FunASR进行中文语音识别 (INT8量化版)"""
    print_section("ASR: 语音识别", 60)
    
    try:
        from funasr import AutoModel
        
        print("正在加载INT8量化ASR模型...")
        print("(首次运行会从ModelScope下载，可能需要几分钟)")
        
        model = AutoModel(
            model="iic/speech_paraformer_asr_nat-zh-cn-16k-common-vocab8358-tensorflow1",
            vad_model=None,
            punc_model=None,
            disable_update=True,
            hub="ms",
            quantize=True,
            device="cpu",
        )
        print("✅ ASR模型加载完成")
        
        if not os.path.exists(audio_path):
            print(f"❌ 音频文件不存在: {audio_path}")
            return ""
        
        print(f"\n处理音频: {audio_path}")
        print("识别中...")
        
        res = model.generate(input=audio_path, batch_size_s=300, language_names=['zh'])
        text = res[0]["text"]
        
        print(f"\n✅ 识别结果: {text}")
        
        # 释放模型
        del model
        force_gc()
        time.sleep(0.5)
        
        return text
    
    except ImportError:
        print("❌ FunASR未安装，请运行: pip install funasr")
        return ""
    except Exception as e:
        print(f"❌ ASR识别失败: {e}")
        return ""


print("✅ ASR模块定义完成")

### 5.2 TTS (文本转语音) 模块 - 快速版本

In [ ]:
"""TTS模块 - MeloTTS (快速版)"""

def tts_synthesis_melo(text: str, output_path: str = None) -> str:
    """使用MeloTTS进行文本转语音 (快速版，<2秒合成)"""
    print_section("TTS: 文本转语音 (MeloTTS)", 60)
    
    if output_path is None:
        output_path = os.path.join(script_dir, 'output', 'synthesis_melo.wav')
    
    try:
        from melo.api import TTS
        
        print("正在加载MeloTTS模型...")
        model = TTS(language="ZH", device="cpu")
        print("✅ MeloTTS模型加载完成")
        
        print(f"\n输入文本: {text}")
        print("合成中...")
        
        speaker_ids = model.hps.data.spk2id
        speaker_id = list(speaker_ids.values())[0]
        
        model.tts_to_file(text, speaker_id, output_path, speed=1.0)
        
        print(f"\n✅ 合成完成")
        print(f"   输出文件: {output_path}")
        
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"   文件大小: {size_mb:.2f} MB")
        
        del model
        force_gc()
        time.sleep(0.5)
        
        return output_path
    
    except ImportError:
        print("❌ MeloTTS未安装，尝试使用edge-tts作为备选...")
        return tts_synthesis_edge_tts(text, output_path)
    except Exception as e:
        print(f"❌ MeloTTS合成失败: {e}")
        return ""


def tts_synthesis_edge_tts(text: str, output_path: str = None) -> str:
    """使用edge-tts进行文本转语音 (在线版，作为备选)"""
    print_section("TTS: 文本转语音 (edge-tts)", 60)
    
    if output_path is None:
        output_path = os.path.join(script_dir, 'output', 'synthesis_edge.wav')
    
    try:
        import subprocess
        
        print(f"\n输入文本: {text}")
        print("合成中 (使用edge-tts)...")
        
        # 使用edge-tts命令行工具
        cmd = ['edge-tts', '--voice', 'zh-CN-YunxiNeural', '--text', text, '--write-media', output_path]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            print(f"\n✅ 合成完成")
            print(f"   输出文件: {output_path}")
            return output_path
        else:
            print(f"❌ edge-tts执行失败: {result.stderr}")
            return ""
    
    except ImportError:
        print("❌ edge-tts未安装")
        return ""
    except Exception as e:
        print(f"❌ edge-tts合成失败: {e}")
        return ""


print("✅ TTS模块定义完成")

### 5.3 音频播放

In [ ]:
"""音频播放"""

def play_audio(audio_path: str):
    """播放音频文件"""
    if not os.path.exists(audio_path):
        print(f"❌ 音频文件不存在: {audio_path}")
        return
    
    try:
        import subprocess
        
        print(f"\n播放音频: {audio_path}")
        
        # 尝试使用不同的播放器
        for player in ['aplay', 'paplay', 'ffplay', 'mpv']:
            try:
                subprocess.run([player, audio_path], timeout=30)
                print(f"✅ 音频已播放")
                return
            except (FileNotFoundError, subprocess.TimeoutExpired):
                continue
        
        print(f"⚠️  无法播放音频 (未找到播放器)")
    except Exception as e:
        print(f"❌ 播放失败: {e}")


print("✅ 音频播放函数定义完成")

### 5.4 完整管道 - 快速版本（MeloTTS）

In [ ]:
"""完整管道演示 - 快速版本"""

def run_full_pipeline_fast(audio_path: str = None, user_input: str = None):
    """
    ASR → LLM → TTS 完整管道 (快速版)
    
    Args:
        audio_path: 输入音频文件 (如果为None，使用user_input)
        user_input: 用户输入文本 (作为备选)
    """
    print_section("完整管道: ASR → LLM → TTS (快速版)", 60)
    
    try:
        # 步骤1: ASR
        if audio_path and os.path.exists(audio_path):
            user_text = asr_recognition_int8(audio_path)
        elif user_input:
            user_text = user_input
            print(f"\n使用提供的文本: {user_text}")
        else:
            print("❌ 需要提供音频文件或文本输入")
            return
        
        if not user_text:
            print("❌ ASR识别失败或文本为空")
            return
        
        # 步骤2: LLM
        if not available_models or _rkllm_lib is None:
            print("⚠️  跳过LLM推理，使用echo作为响应")
            llm_response = f"你说了: {user_text}"
        else:
            print_section("LLM: 智能推理", 60)
            model_path = available_models[0]
            llm = create_rkllm_from_hf(model_path, max_new_tokens=128)
            
            messages = [{"role": "user", "content": user_text}]
            print(f"\n用户: {user_text}")
            print("LLM推理中...")
            
            llm_response = llm.chat(messages)
            print(f"\n助手: {llm_response}")
            
            del llm
            force_gc()
            time.sleep(0.5)
        
        # 步骤3: TTS
        output_audio = tts_synthesis_melo(llm_response)
        
        if output_audio and os.path.exists(output_audio):
            play_audio(output_audio)
        
        print("\n" + "="*60)
        print("✅ 完整管道执行成功")
        print("="*60)
        
    except Exception as e:
        print(f"\n❌ 管道执行失败: {e}")
        import traceback
        traceback.print_exc()


print("✅ 完整管道函数定义完成")

### 5.5 测试：快速管道演示

In [ ]:
"""测试3: 快速管道演示 (不需要真实音频)"""

# 使用文本输入测试完整管道
test_input = "今天天气怎么样?"
run_full_pipeline_fast(user_input=test_input)

## 第6部分：完整管道 - 离线版本（PaddleSpeech）

In [ ]:
"""TTS模块 - PaddleSpeech (离线版)"""

def tts_synthesis_paddle(text: str, output_path: str = None) -> str:
    """使用PaddleSpeech进行文本转语音 (完全离线)"""
    print_section("TTS: 文本转语音 (PaddleSpeech)", 60)
    
    if output_path is None:
        output_path = os.path.join(script_dir, 'output', 'synthesis_paddle.wav')
    
    try:
        from paddlespeech.cli.tts import TTSExecutor
        
        print("正在加载PaddleSpeech模型...")
        tts = TTSExecutor()
        print("✅ PaddleSpeech模型加载完成")
        
        print(f"\n输入文本: {text}")
        print("合成中...")
        
        tts(text=text, output_path=output_path)
        
        print(f"\n✅ 合成完成")
        print(f"   输出文件: {output_path}")
        
        del tts
        force_gc()
        time.sleep(0.5)
        
        return output_path
    
    except ImportError:
        print("❌ PaddleSpeech未安装，请运行: pip install paddlespeech")
        return ""
    except Exception as e:
        print(f"❌ PaddleSpeech合成失败: {e}")
        return ""


print("✅ PaddleSpeech TTS模块定义完成")

In [ ]:
"""完整管道 - 离线版本"""

def run_full_pipeline_offline(audio_path: str = None, user_input: str = None):
    """
    ASR → LLM → TTS 完整管道 (完全离线版)
    
    Args:
        audio_path: 输入音频文件 (如果为None，使用user_input)
        user_input: 用户输入文本 (作为备选)
    """
    print_section("完整管道: ASR → LLM → TTS (离线版)", 60)
    
    try:
        # 步骤1: ASR
        if audio_path and os.path.exists(audio_path):
            user_text = asr_recognition_int8(audio_path)
        elif user_input:
            user_text = user_input
            print(f"\n使用提供的文本: {user_text}")
        else:
            print("❌ 需要提供音频文件或文本输入")
            return
        
        if not user_text:
            print("❌ ASR识别失败或文本为空")
            return
        
        # 步骤2: LLM
        if not available_models or _rkllm_lib is None:
            print("⚠️  跳过LLM推理，使用echo作为响应")
            llm_response = f"你说了: {user_text}"
        else:
            print_section("LLM: 智能推理", 60)
            model_path = available_models[0]
            llm = create_rkllm_from_hf(model_path, max_new_tokens=128)
            
            messages = [{"role": "user", "content": user_text}]
            print(f"\n用户: {user_text}")
            print("LLM推理中...")
            
            llm_response = llm.chat(messages)
            print(f"\n助手: {llm_response}")
            
            del llm
            force_gc()
            time.sleep(0.5)
        
        # 步骤3: TTS (离线版)
        output_audio = tts_synthesis_paddle(llm_response)
        
        if output_audio and os.path.exists(output_audio):
            play_audio(output_audio)
        
        print("\n" + "="*60)
        print("✅ 离线管道执行成功")
        print("="*60)
        
    except Exception as e:
        print(f"\n❌ 管道执行失败: {e}")
        import traceback
        traceback.print_exc()


print("✅ 离线管道函数定义完成")

## 第7部分：模型转换工具

用于在x86 Linux机器上转换HuggingFace模型到RKLLM格式

In [ ]:
"""模型转换工具 - HuggingFace到RKLLM"""

def convert_model_to_rkllm(model_name_or_path: str, output_dir: str = None, quantize: str = "w8a8"):
    """
    将HuggingFace模型转换为RKLLM格式
    
    注意: 此函数需要在x86 Linux机器上运行，需要rkllm-toolkit
    
    Args:
        model_name_or_path: HuggingFace模型名称或本地路径
        output_dir: 输出目录
        quantize: 量化方式 (w8a8, w8a8_asym)
    """
    print_section("模型转换工具", 60)
    
    if output_dir is None:
        output_dir = os.path.join(script_dir, 'models')
    
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n📁 工作目录: {script_dir}")
    print(f"📦 模型: {model_name_or_path}")
    print(f"💾 输出目录: {output_dir}")
    print(f"⚙️  量化方式: {quantize}")
    print(f"🎯 目标平台: rk3588")
    
    print("\n" + "="*60)
    print("转换步骤:")
    print("="*60)
    print("""
1. 安装rkllm-toolkit:
   pip install rkllm-toolkit

2. 下载模型:
   from transformers import AutoTokenizer, AutoModelForCausalLM
   model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
   tokenizer = AutoTokenizer.from_pretrained(model_name)
   model = AutoModelForCausalLM.from_pretrained(model_name)

3. 转换模型:
   from rkllm.api import RKLLM
   
   rkllm = RKLLM(model_path='./model')
   rkllm.build_onnx_quantized(
       do_quantization=True,
       quantization_dtype='int8',
       target_platform='rk3588',
       output_folder='./models'
   )

4. 将生成的.rkllm文件拷贝到RK3588:
   scp models/*.rkllm quark@rk3588:/home/quark/code/models/
    """)
    
    print("\n⚠️  此任务需要在x86 Linux机器上运行")
    print("   转换完成后，将.rkllm文件拷贝到RK3588的models目录")


def generate_calibration_data(model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"):
    """
    生成模型量化所需的校准数据
    
    Args:
        model_name: HuggingFace模型名称
    """
    print_section("生成校准数据", 60)
    
    calibration_texts = [
        "你好，请介绍一下自己。",
        "什么是人工智能？",
        "如何学习编程？",
        "今天天气怎么样？",
        "请帮我解答一个问题。",
        "中国有哪些著名的景点？",
        "如何提高工作效率？",
        "请给我一些建议。",
        "你能做什么？",
        "请讲一个故事。",
    ]
    
    print(f"\n模型: {model_name}")
    print(f"\n校准数据集 ({len(calibration_texts)} 条):")
    for i, text in enumerate(calibration_texts, 1):
        print(f"  {i}. {text}")
    
    print("\n💡 提示: 更多和更长的校准文本会提高量化质量")
    
    return calibration_texts


print("✅ 模型转换工具定义完成")

## 第8部分：总结与快速参考

In [ ]:
"""快速参考和总结"""

print("""
╔═══════════════════════════════════════════════════════════════╗
║            智侠 (ZhiXia) - Notebook功能总结                  ║
╚═══════════════════════════════════════════════════════════════╝

【核心模块】
  - RKLLM推理引擎: NPU加速LLM推理
  - ASR模块: 语音识别 (FunASR)
  - TTS模块: 文本转语音 (MeloTTS/edge-tts/PaddleSpeech)
  - 完整管道: ASR → LLM → TTS

【快速测试函数】

1. 模型检查:
   list_models()                    # 列出可用的RKLLM模型
   check_file_exists(file_path)      # 检查文件是否存在

2. LLM推理:
   llm = create_rkllm_from_hf(model_path)
   response = llm.generate("你好")   # 简单生成
   response = llm.chat(messages)     # 对话模式

3. 语音处理:
   text = asr_recognition_int8(audio_path)              # ASR识别
   audio = tts_synthesis_melo(text)                     # TTS合成
   play_audio(audio)                                    # 播放音频

4. 完整管道:
   run_full_pipeline_fast(user_input="你好")            # 快速版
   run_full_pipeline_offline(user_input="你好")         # 离线版

5. 模型转换 (x86 Linux):
   convert_model_to_rkllm("Qwen/Qwen2.5-1.5B-Instruct")
   calibration_data = generate_calibration_data()

【配置和环境】
  工作目录: {}
  模型目录: {}
  输出目录: {}
  缓存目录: {}

【文件位置】
  该Notebook: {}

【注意事项】
  ⚠️  RKLLM库仅在RK3588上可用，开发机会显示警告
  ⚠️  首次运行模型加载较慢，涉及下载模型文件
  ⚠️  每个模型加载后需要及时释放 (del + force_gc())
  ⚠️  建议最多在内存中保留一个模型实例

【推荐工作流】
  1. 运行第1部分: 基础配置
  2. 运行第2部分: RKLLM基础设施
  3. 运行第4部分: LLM推理测试 (可选)
  4. 运行第5部分: 快速管道测试
  5. 运行第6部分: 离线管道测试 (对比)

╔═══════════════════════════════════════════════════════════════╗
""".format(
    script_dir,
    os.path.join(script_dir, 'models'),
    os.path.join(script_dir, 'output'),
    os.path.join(script_dir, '.cache', 'modelscope'),
    os.path.abspath(__file__) if not 'get_ipython' in dir() else "ZhiXia_Testing_Notebook.ipynb"
))

In [ ]:
# 依赖检查
print("\n【依赖包检查】\n")

dependencies = {
    'torch/pytorch': 'torch',
    'FunASR': 'funasr',
    'MeloTTS': 'melo',
    'PaddleSpeech': 'paddlespeech',
    'edge-tts': 'edge_tts',
    'pyttsx3': 'pyttsx3',
    'faster-whisper': 'faster_whisper',
}

for pkg_name, import_name in dependencies.items():
    try:
        __import__(import_name)
        print(f"  ✅ {pkg_name}")
    except ImportError:
        print(f"  ❌ {pkg_name} (需要安装)")

print("\n💡 安装建议:")
print("   pip install torch funasr melo paddlespeech edge-tts pyttsx3 faster-whisper")